# Local Inference and API Server Setup

This notebook covers:

1. Building llama.cpp inside Termux.
2. Running local inference.
3. Benchmarking model performance.
4. Starting an OpenAI-compatible API server.

## 1. Clone llama.cpp


```bash
git clone https://github.com/ggml-org/llama.cpp.git
cd llama.cpp


## 2. Build Dependencies

```bash
pkg install git cmake clang make python -y

## 3. Configure Build
The default configuration may fail on newer Snapdragon CPUs due to LLVM backend crashes.


```bash
rm -rf build

cmake -B build \
  -DGGML_NATIVE=OFF \
  -DGGML_CPU_ALL_VARIANTS=OFF

## 4. Build llama.cpp

```bash
cmake --build build -j4

## 5. Troubleshooting

### LLVM Crash During Build

Example:

clang frontend command failed with exit code 139

Cause:
LLVM code generation failure when compiling ARM SVE/SME optimized kernels on certain Snapdragon devices.

Resolution:
```bash
cmake -B build \
  -DGGML_NATIVE=OFF \
  -DGGML_CPU_ALL_VARIANTS=OFF
```

This disables problematic native optimization paths and allows compilation to succeed.

## 6. Verify binaries


Expected:
```
llama-cli
llama-server
llama-bench
```

## 7. Verify model

```bash
ls ~/storage/downloads

Expected:
`qwen2.5-coder-7b-q4_0.gguf`

## 8. First Inference

```bash
./build/bin/llama-cli \
  -m ~/storage/downloads/qwen2.5-coder-7b-q4_0.gguf \
  -n 64 \
  -p "Write a Python hello world program"
```

## 9. Benchmark

```bash
./build/bin/llama-bench \
  -m ~/storage/downloads/qwen2.5-coder-7b-q4_0.gguf
```

Benchmark Result:
```
Prompt Processing: ~9 tok/s
Generation: ~6.4 tok/s
Device: Samsung Galaxy S26 Ultra
Backend: CPU
Model: Qwen2.5-Coder-7B Q4_0
```

## 10. Start server

```bash
./build/bin/llama-server \
  -m ~/storage/downloads/qwen2.5-coder-7b-q4_0.gguf \
  --host 0.0.0.0 \
  --port 8080 \
  -c 2048
  ```

### Find IP

```bash
ip addr show wlan0

Look for: `192.168.x.x`

## 11. Verify API from laptop

```bash
curl http://PHONE_IP:8080/v1/models

Expected:
```json
{
  "data": [...]
}

## 12. Completion Criteria

The notebook is complete when:

- llama.cpp builds successfully.
- The model loads successfully.
- Benchmarking completes.
- llama-server starts successfully.
- The OpenAI-compatible API endpoint is reachable from another device.